In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LocalSpark") \
    .getOrCreate()

data = [("Dev", 100), ("Aryan", 200)]

df = spark.createDataFrame(data, ["name", "score"])

df.show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/04 19:03:00 WARN Utils: Your hostname, Devs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.101 instead (on interface en0)
26/05/04 19:03:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/04 19:03:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

+-----+-----+
| name|score|
+-----+-----+
|  Dev|  100|
|Aryan|  200|
+-----+-----+



In [2]:
pwd

'/Users/devbhandari/DevB/Learning/Study/spark-learning'

In [3]:
print('hello databricks')

hello databricks


#### Read CSV Using Spark

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lakehouse") \
    .getOrCreate()

df = spark.read.csv(
    "data/raw/orders.csv",
    header=True,
    inferSchema=True
)

df.show()

26/05/04 19:19:26 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+--------+--------+-------+------+
|order_id|customer|country|amount|
+--------+--------+-------+------+
|       1|     Dev|  India|   100|
|       2|   Aryan|    USA|   200|
|       3|   Rahul|  India|   150|
|       4|    John|     UK|   300|
|       5|     Dev|  India|   120|
+--------+--------+-------+------+



Bronze Layer

In [12]:
# Write Bronze
df.write.mode("overwrite").parquet("data/bronze/orders")

In [13]:
# Read Bronze:
bronze_df = spark.read.parquet("data/bronze/orders")

bronze_df.show()

+--------+--------+-------+------+
|order_id|customer|country|amount|
+--------+--------+-------+------+
|       1|     Dev|  India|   100|
|       2|   Aryan|    USA|   200|
|       3|   Rahul|  India|   150|
|       4|    John|     UK|   300|
|       5|     Dev|  India|   120|
+--------+--------+-------+------+



### Silver Layer

In [16]:
from pyspark.sql.functions import col

silver_df = bronze_df.filter(
    col("amount") > 100
)

silver_df.show()

+--------+--------+-------+------+
|order_id|customer|country|amount|
+--------+--------+-------+------+
|       2|   Aryan|    USA|   200|
|       3|   Rahul|  India|   150|
|       4|    John|     UK|   300|
|       5|     Dev|  India|   120|
+--------+--------+-------+------+



In [18]:
#write silver
silver_df.write.mode("overwrite").parquet("data/silver/orders")

In [19]:
#Read silver
silver_df = spark.read.parquet("data/silver/orders")

silver_df.show()

+--------+--------+-------+------+
|order_id|customer|country|amount|
+--------+--------+-------+------+
|       2|   Aryan|    USA|   200|
|       3|   Rahul|  India|   150|
|       4|    John|     UK|   300|
|       5|     Dev|  India|   120|
+--------+--------+-------+------+



### Gold Layer

In [20]:
gold_df = silver_df.groupBy("country").sum("amount")

gold_df.show()

+-------+-----------+
|country|sum(amount)|
+-------+-----------+
|  India|        270|
|    USA|        200|
|     UK|        300|
+-------+-----------+



### Write Gold

In [21]:
gold_df.write.mode("overwrite").parquet("data/gold/orders")

In [24]:
##Read Gold
gold_df = spark.read.parquet("data/gold/orders")

gold_df.show()

+-------+-----------+
|country|sum(amount)|
+-------+-----------+
|  India|        270|
|    USA|        200|
|     UK|        300|
+-------+-----------+



In [1]:
import sys
print(sys.executable)

/Users/devbhandari/DevB/Learning/Study/spark-learning/venv/bin/python


In [3]:
# Test from vs code
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("VSCodeSpark") \
    .getOrCreate()

data = [("Dev", 100), ("Aryan", 200)]

df = spark.createDataFrame(data, ["name", "score"])

df.show()

The operation couldn’t be completed. Unable to locate a Java Runtime.
Please visit http://www.java.com for information on installing Java.

/Users/devbhandari/DevB/Learning/Study/spark-learning/venv/lib/python3.11/site-packages/pyspark/bin/spark-class: line 97: CMD: bad array subscript
head: illegal line count -- -1


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.